In [7]:
from pathlib import Path
import pandas as pd
import yaml

ZEEK_DIR = Path("zeek_logs")
META_PATH = Path("attack_metadata.yaml")
OUTPUT_PATH = Path("ciciot2023_labeled_conn.tsv")

In [8]:
with open(META_PATH, "r") as f:
    metadata = yaml.safe_load(f)

In [9]:
def get_ips_for_macs(mapping_path: Path, victim_macs: list[str]) -> set[str]:
    if not mapping_path.exists():
        return set()

    if not victim_macs:
        return set()

    macs = {
        str(mac).lower().strip()
        for mac in victim_macs
        if str(mac).strip() and str(mac).strip() != "-"
    }

    if not macs:
        return set()

    mapping = pd.read_csv(mapping_path, sep="\t")

    if mapping.empty:
        return set()

    mapping["mac"] = mapping["mac"].astype(str).str.lower().str.strip()

    rows = mapping[mapping["mac"].isin(macs)]

    victim_ips = set()

    for value in rows["ip_addresses"].dropna():
        for ip in str(value).split(";"):
            ip = ip.strip()
            if ip and ip != "-":
                victim_ips.add(ip)

    return victim_ips

In [13]:
all_labeled = []
summary_rows = []

for folder in sorted(ZEEK_DIR.iterdir()):
    if not folder.is_dir():
        continue

    stem = folder.name
    conn_path = folder / "conn.tsv"
    mapping_path = folder / "mac_ip_mapping.tsv"

    if not conn_path.exists():
        print(f"Skipping {stem}: missing conn.tsv")
        continue

    if stem not in metadata:
        print(f"Skipping {stem}: no entry in attack_metadata.yaml")
        continue

    attack_label = metadata[stem]["label"]
    victim_macs = metadata[stem].get("victim_macs", [])
    known_ips = metadata[stem].get("known_ips", [])

    known_ips = {
        str(ip).strip()
        for ip in known_ips
        if str(ip).strip() and str(ip).strip() != "-"
    }

    victim_ips = get_ips_for_macs(mapping_path, victim_macs)
    victim_ips.update(known_ips)

    conn = pd.read_csv(conn_path, sep="\t", low_memory=False)

    if attack_label == "BENIGN":
        conn["label"] = "BENIGN"
    
    elif attack_label == "BRUTE_FORCE":
        conn["label"] = "BRUTE_FORCE"
    elif attack_label == "SQL_INJECTION":
        conn["label"] = "SQL_INJECTION"
    elif attack_label == "XSS":
        conn["label"] = "XSS"

    elif not victim_ips:
        conn["label"] = "BACKGROUND"
        print(f"Warning: {stem}: no victim IPs found from victim_macs or known_ips")

    else:
        victim_mask = (
            conn["id.resp_h"].isin(victim_ips)
        )

        conn["label"] = "BACKGROUND"
        conn.loc[victim_mask, "label"] = attack_label

    summary_rows.append({
        "pcap_stem": stem,
        "attack_label": attack_label,
        "victim_macs": ";".join(victim_macs),
        "known_ips": ";".join(sorted(known_ips)) if known_ips else "-",
        "resolved_victim_ips": ";".join(sorted(victim_ips)) if victim_ips else "-",
        "total_flows": len(conn),
        "attack_labeled_flows": int((conn["label"] == attack_label).sum()) if attack_label != "BENIGN" else 0,
        "background_flows": int((conn["label"] == "BACKGROUND").sum()),
        "benign_flows": int((conn["label"] == "BENIGN").sum()),
    })

    all_labeled.append(conn)

summary = pd.DataFrame(summary_rows)
summary

Skipping BenignTraffic1: no entry in attack_metadata.yaml
Skipping BenignTraffic2: no entry in attack_metadata.yaml
Skipping BenignTraffic3: no entry in attack_metadata.yaml


,pcap_stem,attack_label,victim_macs,known_ips,resolved_victim_ips,total_flows,attack_labeled_flows,background_flows,benign_flows
0,BenignTraffic,BENIGN,,-,-,160231,0,0,160231
1,DictionaryBruteForce,BRUTE_FORCE,44:01:BB:EC:10:4A;7C:A7:B0:CD:18:32;DC:A6:32:C...,192.168.137.131;192.168.137.141;192.168.137.71...,192.168.137.131;192.168.137.141;192.168.137.71...,7658,7658,0,0
2,DoS-HTTP_Flood,DOS_HTTP_FLOOD,2C:71:FF:05:F1:15;9C:8E:CD:1D:AB:9F;B0:C5:54:5...,192.168.137.131;192.168.137.141;192.168.137.17...,192.168.137.131;192.168.137.141;192.168.137.17...,859592,839184,20408,0
3,DoS-HTTP_Flood1,DOS_HTTP_FLOOD,2C:71:FF:05:F1:15;9C:8E:CD:1D:AB:9F;B0:C5:54:5...,192.168.137.131;192.168.137.141;192.168.137.17...,192.168.137.131;192.168.137.141;192.168.137.17...,648997,577776,71221,0
4,Recon-PortScan,PORTSCAN,28:6D:97:9E:F4:D5;1C:FE:2B:98:16:DD;A0:D0:DC:C...,192.168.137.100;192.168.137.110;192.168.137.11...,192.168.137.100;192.168.137.106;192.168.137.11...,216533,45869,170664,0
5,SqlInjection,SQL_INJECTION,DC:A6:32:C9:E6:F4;DC:A6:32:C9:E4:C6;DC:A6:32:C...,192.168.137.131;192.168.137.141;192.168.137.71,192.168.137.131;192.168.137.141;192.168.137.71,5598,5598,0,0
6,XSS,XSS,DC:A6:32:C9:E6:F4;DC:A6:32:C9:E4:C6;DC:A6:32:C...,192.168.137.131;192.168.137.141;192.168.137.71,192.168.137.131;192.168.137.141;192.168.137.71,3347,3347,0,0


In [15]:
final_df = pd.concat(all_labeled, ignore_index=True)
print(final_df["label"].value_counts())

label
DOS_HTTP_FLOOD    1416960
BACKGROUND         262293
BENIGN             160231
PORTSCAN            45869
BRUTE_FORCE          7658
SQL_INJECTION        5598
XSS                  3347
Name: count, dtype: int64


In [16]:
final_df.to_csv(OUTPUT_PATH, sep="\t", index=False)

print(f"Saved {len(final_df):,} rows to {OUTPUT_PATH}")

Saved 1,901,956 rows to ciciot2023_labeled_conn.tsv
